# Amazon Reviews 2023 — Preprocessing (Clean Only)

**Goal:** Produce a single, clean, canonical dataset from `amazon_reviews_2023.csv` for downstream agents.

**We will:**
- Load raw CSV and run basic sanity checks
- Drop duplicates and handle missing values
- Normalize timestamp and booleans
- Light text cleanup for `title` and `text`
- Add simple derived fields (e.g., word counts)
- Save cleaned dataset to `data/processed/amazon_reviews_clean.(csv|parquet)`


In [14]:
import os
import re
import numpy as np
import pandas as pd

RAW_CSV = "Amazon_reviews_2023.csv"               # adjust if needed
OUT_DIR = os.path.join("data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)

pd.set_option("display.max_columns", 200)

def clean_text(s: str) -> str:
    """Light text cleanup: lowercase, strip tags/urls, normalize whitespace."""
    if not isinstance(s, str): 
        return ""
    s = s.lower()
    s = re.sub(r"<[^>]+>", " ", s)            # remove simple HTML tags
    s = re.sub(r"http\S+|www\.\S+", " ", s)   # URLs -> space
    s = re.sub(r"[\r\n\t]+", " ", s)          # newlines/tabs -> space
    s = re.sub(r"\s+", " ", s).strip()
    return s


## Load & Quick Checks

Load the raw CSV, inspect shape and minimal schema assumptions.


In [15]:
df = pd.read_csv(RAW_CSV)
print("Raw shape:", df.shape)
df.head(3)


Raw shape: (701528, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True


## Basic Cleaning
- Drop full-row duplicates
- Handle missing `title`/`text`
- Normalize `timestamp`
- Ensure `verified_purchase` is 0/1
- Clip extreme `helpful_vote` outliers (optional)


In [16]:
# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Dropped duplicates: {before - len(df)}")

# Fill minimal NA for text fields
df["title"] = df["title"].fillna("")
df["text"]  = df["text"].fillna("")

# Normalize timestamp (object → datetime; auto-detect ms vs s)
ts = pd.to_numeric(df["timestamp"], errors="coerce")
unit = "ms" if np.nanmedian(ts) and np.nanmedian(ts) > 10**12 else "s"
df["timestamp"] = pd.to_datetime(ts, unit=unit, errors="coerce")
dropped_ts = df["timestamp"].isna().sum()
print(f"Rows with invalid timestamp: {dropped_ts}")

# Verified purchase to int (0/1)
if df["verified_purchase"].dtype != np.int64 and df["verified_purchase"].dtype != np.int32:
    df["verified_purchase"] = df["verified_purchase"].astype(int)

# Clip extreme helpful votes (optional; 99.9th percentile)
upper = df["helpful_vote"].quantile(0.999)
df["helpful_vote"] = df["helpful_vote"].clip(lower=0, upper=upper)

# Drop rows with invalid timestamp (optional—keep if you need all rows)
df = df.dropna(subset=["timestamp"]).reset_index(drop=True)

print("Post-cleaning shape:", df.shape)
df.head(3)


Dropped duplicates: 7275
Rows with invalid timestamp: 694253
Post-cleaning shape: (0, 10)


/tmp/ipykernel_59340/1036595497.py:12: RuntimeWarning: All-NaN slice encountered
  unit = "ms" if np.nanmedian(ts) and np.nanmedian(ts) > 10**12 else "s"


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase


## Text Normalization & Derived Fields
- Light text cleanup on `title` and `text`
- Compute word counts to help downstream components


In [17]:
df["title_clean"] = df["title"].apply(clean_text)
df["text_clean"]  = df["text"].apply(clean_text)

df["title_words"] = df["title_clean"].apply(lambda s: len(s.split()))
df["text_words"]  = df["text_clean"].apply(lambda s: len(s.split()))

# Optionally trim extreme long reviews to reduce outliers in agents
cutoff = 1500
before = len(df)
df = df[df["text_words"] <= cutoff].copy()
print(f"Trimmed extreme long reviews (> {cutoff} words): {before - len(df)}")

df[["title_words","text_words"]].describe()


Trimmed extreme long reviews (> 1500 words): 0


,title_words,text_words
count,0,0
unique,0,0
top,NaN,NaN
freq,NaN,NaN


## Save Cleaned Dataset
Keep fields useful for agents and analytics, then save CSV/Parquet to `data/processed/`.


In [18]:
keep_cols = [
    "asin", "parent_asin", "user_id",
    "rating", "helpful_vote", "verified_purchase", "timestamp",
    "title_clean", "text_clean", "title_words", "text_words"
]

clean = df[keep_cols].copy()
csv_path = os.path.join(OUT_DIR, "amazon_reviews_clean.csv")
pq_path  = os.path.join(OUT_DIR, "amazon_reviews_clean.parquet")

clean.to_csv(csv_path, index=False)
clean.to_parquet(pq_path, index=False)

print("Saved:")
print(" -", csv_path)
print(" -", pq_path)
print("Final shape:", clean.shape)


Saved:
 - data/processed/amazon_reviews_clean.csv
 - data/processed/amazon_reviews_clean.parquet
Final shape: (0, 11)


### Output
A single, canonical cleaned dataset suitable for your multi-agent pipeline:
- `data/processed/amazon_reviews_clean.csv`
- `data/processed/amazon_reviews_clean.parquet`

This file has normalized timestamps, cleaned text, consistent types, and basic derived features for downstream use.
